In [31]:
!pip install wrds

  Using cached packaging-24.2-py3-none-any.whl.metadata (3.2 kB)
Using cached packaging-24.2-py3-none-any.whl (65 kB)
  Attempting uninstall: packaging
    Found existing installation: packaging 25.0
    Uninstalling packaging-25.0:
ERROR: Could not install packages due to an OSError: Cannot move the non-empty directory '/private/var/folders/fj/plbn4w9d6fs55651zh_pxsd80000gn/T/AppTranslocation/F913C86F-717B-4630-9271-6967A172BCEA/d/Positron 2.app/Contents/Resources/app/extensions/positron-python/python_files/lib/ipykernel/py3/packaging-25.0.dist-info/': Lacking write permission to '/private/var/folders/fj/plbn4w9d6fs55651zh_pxsd80000gn/T/AppTranslocation/F913C86F-717B-4630-9271-6967A172BCEA/d/Positron 2.app/Contents/Resources/app/extensions/positron-python/python_files/lib/ipykernel/py3/packaging-25.0.dist-info/'.



In [32]:
import os
from typing import List, Dict, Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import wrds  # %pip install wrds pyarrow fastparquet

# ---- user-configurable ----
START_DATE = "2000-01-01"
END_DATE   = "2025-08-31"
TOP_N      = 500
ETF_TICKERS = ["SPY", "IWM", "QQQ"]  # change if needed
DATA_DIR   = "./data"
FIG_DIR    = "./figs"

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

In [33]:
def _parquet_path(name: str) -> str:
    return os.path.join(DATA_DIR, f"{name}.parquet")

def cache_df(df: pd.DataFrame, name: str) -> None:
    df.to_parquet(_parquet_path(name), index=False)

def load_cached(name: str) -> Optional[pd.DataFrame]:
    path = _parquet_path(name)
    return pd.read_parquet(path) if os.path.exists(path) else None

In [34]:
def wrds_connect() -> wrds.Connection:
    print("Connecting to WRDS...")
    db = wrds.Connection(wrds_username = "zkg232")
    print("Connected.")
    return db

db = wrds_connect()

Connecting to WRDS...
Loading library list...
Done
Connected.


In [ ]:
def pull_crsp_monthly_panel(db: wrds.Connection, start: str, end: str) -> pd.DataFrame:
    """
    MSF joined to MSENAMES (time-bounded) to bring shrcd/exchcd; filter to:
      shrcd in (10,11)  # U.S. common shares
      exchcd in (1,2,3) # NYSE/AMEX/NASDAQ
    Also clean PRC sign, SHROUT (thousands→shares), compute MKT CAP.
    """
    cache_name = f"msf_panel_{start}_{end}".replace("-", "")
    cached = load_cached(cache_name)
    if cached is not None:
        return cached

    start_y, end_y = pd.to_datetime(start).year, pd.to_datetime(end).year
    frames = []
    for y in range(start_y, end_y + 1):
        chunk_start = f"{y}-01-01" if y > start_y else start
        chunk_end   = f"{y}-12-31" if y < end_y   else end
        q = f"""
            SELECT m.date, m.permno, m.prc, m.shrout, m.ret,
                   n.shrcd, n.exchcd
            FROM crsp.msf AS m
            JOIN crsp.msenames AS n
              ON m.permno = n.permno
             AND m.date BETWEEN n.namedt AND COALESCE(n.nameendt, DATE '9999-12-31')
            WHERE m.date BETWEEN '{chunk_start}' AND '{chunk_end}'
              AND n.shrcd IN (10, 11)
              AND n.exchcd IN (1, 2, 3)
              AND m.prc IS NOT NULL
              AND m.shrout IS NOT NULL
            ORDER BY m.date, m.permno;
        """
        frames.append(db.raw_sql(q))

    df = pd.concat(frames, ignore_index=True)

    df["date"]   = pd.to_datetime(df["date"])
    df["prc"]    = pd.to_numeric(df["prc"], errors="coerce").abs()
    df["shrout"] = pd.to_numeric(df["shrout"], errors="coerce") * 1000.0
    df["ret"]    = pd.to_numeric(df["ret"], errors="coerce")
    df["mktcap"] = df["prc"] * df["shrout"]

    # (df, cache_name)
    return df

In [36]:
def pull_crsp_delistings(db: wrds.Connection, start: str, end: str) -> pd.DataFrame:
    """
    Delisting returns from CRSP.MSEDELIST.
    Correct column is dlstdt (delisting date). We alias to 'date' for easy merging.
    """
    cache_name = f"delist_{start}_{end}".replace("-", "")
    cached = load_cached(cache_name)
    if cached is not None:
        return cached

    q = f"""
        SELECT permno,
               dlstcd,
               dlstdt AS date,
               dlret
        FROM crsp.msedelist
        WHERE dlstdt BETWEEN '{start}' AND '{end}';
    """
    d = db.raw_sql(q)
    d["date"]  = pd.to_datetime(d["date"])
    d["dlret"] = pd.to_numeric(d["dlret"], errors="coerce")
    # cache_df(d, cache_name)
    return d

In [37]:
def add_effective_returns(panel: pd.DataFrame, delist: pd.DataFrame) -> pd.DataFrame:
    """
    reff = (1 + ret) * (1 + dlret) - 1
    Merge by (permno, date) where 'date' is the month and delist month if any.
    """
    out = panel.merge(delist[["permno", "date", "dlret"]], on=["permno", "date"], how="left")
    out["ret_eff"] = (1.0 + out["ret"].fillna(0.0)) * (1.0 + out["dlret"].fillna(0.0)) - 1.0
    return out

In [38]:
def build_topn_indexes(panel: pd.DataFrame, top_n: int) -> pd.DataFrame:
    """
    Base=100; for month t, select members & weights from t-1, apply ret_eff over t.
    Carry forward levels if data incomplete.
    """
    panel = panel.sort_values(["date", "permno"])
    months = panel["date"].drop_duplicates().sort_values().tolist()
    by_month = {d: g for d, g in panel.groupby("date")}

    ew_level = [100.0]
    vw_level = [100.0]
    pw_level = [100.0]

    for i in range(1, len(months)):
        t_1, t = months[i-1], months[i]
        g_t1 = by_month[t_1].dropna(subset=["mktcap", "prc"])
        g_t  = by_month[t]

        if g_t1.empty or g_t.empty:
            ew_level.append(ew_level[-1]); vw_level.append(vw_level[-1]); pw_level.append(pw_level[-1])
            continue

        chosen = g_t1.sort_values("mktcap", ascending=False).head(top_n).set_index("permno")
        g_t = g_t.set_index("permno")
        common = chosen.index.intersection(g_t.index)
        if len(common) == 0:
            ew_level.append(ew_level[-1]); vw_level.append(vw_level[-1]); pw_level.append(pw_level[-1])
            continue

        r_t = g_t.loc[common, "ret_eff"].astype(float)

        # EW
        r_t_ew = r_t.dropna()
        ew_gross = (1.0 + r_t_ew).mean() if not r_t_ew.empty else 1.0

        # VW (weights ∝ t-1 mktcap)
        w_vw = chosen.loc[common, "mktcap"].astype(float)
        mask_vw = r_t.notna()
        w_vw = w_vw.loc[mask_vw]; r_vw = r_t.loc[mask_vw]
        if w_vw.empty or w_vw.sum() <= 0:
            vw_gross = 1.0
        else:
            w_vw = w_vw / w_vw.sum()
            vw_gross = ((1.0 + r_vw) * w_vw).sum()

        # PW (weights ∝ t-1 price)
        w_pw = chosen.loc[common, "prc"].astype(float)
        mask_pw = r_t.notna()
        w_pw = w_pw.loc[mask_pw]; r_pw = r_t.loc[mask_pw]
        if w_pw.empty or w_pw.sum() <= 0:
            pw_gross = 1.0
        else:
            w_pw = w_pw / w_pw.sum()
            pw_gross = ((1.0 + r_pw) * w_pw).sum()

        ew_level.append(ew_level[-1] * float(ew_gross))
        vw_level.append(vw_level[-1] * float(vw_gross))
        pw_level.append(pw_level[-1] * float(pw_gross))

    idx_months = pd.to_datetime(months[1:])
    ew = pd.Series(ew_level[1:], index=idx_months, name="Equal-Weighted")
    vw = pd.Series(vw_level[1:], index=idx_months, name="Value-Weighted")
    pw = pd.Series(pw_level[1:], index=idx_months, name="Price-Weighted")
    return pd.concat([ew, vw, pw], axis=1)

In [39]:
def pick_permno_for_ticker(db: wrds.Connection, ticker: str, ref_date: str) -> Optional[int]:
    # note: nameenddt (not nameendt)
    q = f"""
        SELECT permno, ticker, namedt, nameenddt
        FROM crsp.stocknames
        WHERE ticker = '{ticker}'
        ORDER BY permno, namedt;
    """
    names = db.raw_sql(q)
    if names.empty:
        return None

    ref_dt = pd.to_datetime(ref_date)
    names["namedt"]    = pd.to_datetime(names["namedt"])
    # nameenddt can be null for still-active rows
    names["nameenddt"] = pd.to_datetime(names["nameenddt"]).fillna(pd.Timestamp("2099-12-31"))

    # prefer a row whose name window covers ref_date
    covering = names[(names["namedt"] <= ref_dt) & (ref_dt <= names["nameenddt"])]
    if not covering.empty:
        return int(covering.iloc[0]["permno"])

    # otherwise take the one with namedt closest before ref_date, else earliest
    before = names[names["namedt"] <= ref_dt].copy()
    if not before.empty:
        before["diff"] = (ref_dt - before["namedt"]).dt.days
        before = before.sort_values("diff")
        return int(before.iloc[0]["permno"])

    return int(names.sort_values("namedt").iloc[0]["permno"])


def get_etf_index_series(db: wrds.Connection, permno: int, start: str, end: str, label: str) -> pd.Series:
    q = f"""
        SELECT date, ret
        FROM crsp.msf
        WHERE permno = {permno}
          AND date BETWEEN '{start}' AND '{end}'
        ORDER BY date;
    """
    d = db.raw_sql(q)
    d["date"] = pd.to_datetime(d["date"]).sort_values()
    d = d.sort_values("date")
    d["ret"] = pd.to_numeric(d["ret"], errors="coerce").fillna(0.0)
    idx = (1.0 + d["ret"]).cumprod() * 100.0
    idx.index = d["date"]
    idx.name = label
    return idx


def pull_benchmarks(db: wrds.Connection, tickers: List[str], start: str, end: str) -> Dict[str, pd.Series]:
    series = {}
    for tkr in tickers:
        pno = pick_permno_for_ticker(db, tkr, start)
        if pno is None:
            print(f"Warning: no PERMNO found for {tkr} near {start}. Skipping.")
            continue
        series[tkr] = get_etf_index_series(db, pno, start, end, f"{tkr} (ETF)")
    return series

In [40]:
def align_panel_for_corr(levels: List[pd.Series]) -> pd.DataFrame:
    if not levels:
        return pd.DataFrame()
    common = set(levels[0].index)
    for s in levels[1:]:
        common &= set(s.index)
    common = sorted(common)
    aligned = [s.loc[common] for s in levels]
    panel = pd.concat(aligned, axis=1)
    log_ret = np.log(panel / panel.shift(1)).dropna(how="any")
    return log_ret

In [41]:
def plot_index_levels(custom_df: pd.DataFrame, etf_map: Dict[str, pd.Series], top_n: int, outfile: str) -> None:
    plt.figure(figsize=(10, 6))
    for col in custom_df.columns:
        custom_df[col].plot(label=col)
    for k, s in etf_map.items():
        s.plot(label=k)
    plt.title(f"Custom Top-{top_n} Indexes vs. ETFs")
    plt.xlabel("Date")
    plt.ylabel("Index Level (Base = 100)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(outfile, dpi=220)
    plt.close()

def plot_corr_heatmap(corr: pd.DataFrame, outfile: str) -> None:
    if corr.empty:
        return
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(corr.values, aspect="auto")
    ax.set_xticks(range(corr.shape[1])); ax.set_yticks(range(corr.shape[0]))
    ax.set_xticklabels(corr.columns, rotation=45, ha="right")
    ax.set_yticklabels(corr.index)
    for i in range(corr.shape[0]):
        for j in range(corr.shape[1]):
            ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=9)
    plt.title("Correlation Matrix (Monthly Log Returns)")
    fig.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.savefig(outfile, dpi=220)
    plt.close()

In [42]:
print("[1/7] Pull CRSP panel")
panel = pull_crsp_monthly_panel(db, START_DATE, END_DATE)
print(f"Panel rows: {len(panel):,}")
display(panel.head())

print("[2/7] Pull delistings")
delist = pull_crsp_delistings(db, START_DATE, END_DATE)
print(f"Delist rows: {len(delist):,}")
display(delist.head())

print("[3/7] Effective returns (ret_eff)")
panel = add_effective_returns(panel, delist)
display(panel[["date","permno","ret","dlret","ret_eff"]].head())

print("[4/7] Build custom EW/VW/PW indexes (Top-N =", TOP_N, ")")
custom_levels = build_topn_indexes(panel, TOP_N)
display(custom_levels.tail())

print("[5/7] ETF benchmarks")
etf_series = pull_benchmarks(db, ETF_TICKERS, START_DATE, END_DATE)
for k, s in etf_series.items():
    print(k, "length:", len(s))

print("[6/7] Plots")
levels_png = os.path.join(FIG_DIR, f"indexes_top{TOP_N}.png")
plot_index_levels(custom_df=custom_levels, etf_map=etf_series, top_n=TOP_N, outfile=levels_png)
print("Saved:", levels_png)

print("[7/7] Correlations & summary table")
all_series = [custom_levels[c] for c in custom_levels.columns] + [s for s in etf_series.values()]
logret_df  = align_panel_for_corr(all_series)
corr_mat   = logret_df.corr()
corr_png   = os.path.join(FIG_DIR, f"corr_top{TOP_N}.png")
plot_corr_heatmap(corr_mat, corr_png)
print("Saved:", corr_png)
display(corr_mat)

# quick performance table for appendix (optional)
summ = pd.DataFrame({
    "mean_log_ret": logret_df.mean(),
    "std_log_ret": logret_df.std(ddof=1),
})
summ["ann_mean_log_ret"] = summ["mean_log_ret"] * 12.0
summ["ann_std_log_ret"]  = summ["std_log_ret"] * np.sqrt(12.0)
summ["sharpe_log"]       = np.where(summ["ann_std_log_ret"]>0,
                                    summ["ann_mean_log_ret"]/summ["ann_std_log_ret"], np.nan)
display(summ.round(4))

[1/7] Pull CRSP panel


<positron-console-cell-42>:34: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.


ArrowKeyError: A type extension with name pandas.period already defined